In [0]:
# COMPLETE PROJECT VALIDATION
from pyspark.sql import functions as F
print("Catalog : retail_demo")
print("Mode    : Read-only validation")

Catalog : retail_demo
Mode    : Read-only validation


In [0]:

# VALIDATE ACTUAL TABLES CREATED BY THE PROJECT

raw_tables = (
    spark.sql("SHOW TABLES IN retail_demo.raw")
    .select("tableName", "isTemporary")
    .filter(F.col("isTemporary") == False)
)

print("\nAVAILABLE RAW TABLES:")
display(
    raw_tables.orderBy("tableName")
)
actual_bronze_tables = [
    "customers_bronze",
    "orders_bronze",
    "products_bronze",
    "stores_bronze",
    "bronze_customers_incremental",
    "bronze_orders_incremental",
    "bronze_products_incremental"
]
bronze_results = []

for table_name in actual_bronze_tables:

    full_name = f"retail_demo.raw.{table_name}"

    try:

        df = spark.table(full_name)

        row_count = df.count()
        column_count = len(df.columns)

        bronze_results.append(
            (
                table_name,
                row_count,
                column_count,
                "PASS"
            )
        )

    except Exception as e:

        bronze_results.append(
            (
                table_name,
                None,
                None,
                f"ERROR: {str(e)[:150]}"
            )
        )

for (
    table_name,
    row_count,
    column_count,
    status
) in bronze_results:

    if status == "PASS":

        print(
            f"✓ retail_demo.raw.{table_name:<35}"
            f"{row_count:,} rows | "
            f"{column_count} columns"
        )

    else:

        print(
            f"✗ retail_demo.raw.{table_name:<35}"
            f"{status}"
        )
for table_name, row_count, column_count, status in bronze_results:

    if status == "PASS":
        print(
            f"{table_name:<35} : {row_count:,}"
        )

print("\n" + "=" * 80)
print("BRONZE VALIDATION COMPLETE")
print("=" * 80)


AVAILABLE RAW TABLES:


tableName,isTemporary
bronze_customers_incremental,false
bronze_orders_incremental,false
bronze_products_incremental,false
customers_bronze,false
orders_bronze,false
products_bronze,false
stores_bronze,false


✓ retail_demo.raw.customers_bronze                   2,560 rows | 10 columns
✓ retail_demo.raw.orders_bronze                      12,180 rows | 14 columns
✓ retail_demo.raw.products_bronze                    830 rows | 10 columns
✓ retail_demo.raw.stores_bronze                      80 rows | 8 columns
✓ retail_demo.raw.bronze_customers_incremental       630 rows | 29 columns
✓ retail_demo.raw.bronze_orders_incremental          6,135 rows | 17 columns
✓ retail_demo.raw.bronze_products_incremental        180 rows | 13 columns
customers_bronze                    : 2,560
orders_bronze                       : 12,180
products_bronze                     : 830
stores_bronze                       : 80
bronze_customers_incremental        : 630
bronze_orders_incremental           : 6,135
bronze_products_incremental         : 180

BRONZE VALIDATION COMPLETE


In [0]:
# SILVER STAGE 1 VALIDATION=
silver_stage1_tables = {
    "silver1_customers_clean":
        "retail_demo.silver.silver1_customers_clean",

    "silver1_products_clean":
        "retail_demo.silver.silver1_products_clean",

    "silver1_orders_clean":
        "retail_demo.silver.silver1_orders_clean",

    "silver1_stores_clean":
        "retail_demo.silver.silver1_stores_clean",

    "quarantine_orders":
        "retail_demo.silver.quarantine_orders"
}

for label, table_name in silver_stage1_tables.items():

    try:

        df = spark.table(table_name)

        row_count = df.count()
        column_count = len(df.columns)

        print(
            f"✓ {table_name:<55}"
            f"{row_count:,} rows | "
            f"{column_count} columns"
        )

    except Exception as e:

        print(
            f"✗ {table_name:<55}"
            f"ERROR: {str(e)[:150]}"
        )
display(
    spark.sql("""
        SELECT
            'silver1_customers_clean' AS table_name,
            COUNT(*) AS row_count
        FROM retail_demo.silver.silver1_customers_clean

        UNION ALL

        SELECT
            'silver1_products_clean',
            COUNT(*)
        FROM retail_demo.silver.silver1_products_clean

        UNION ALL

        SELECT
            'silver1_orders_clean',
            COUNT(*)
        FROM retail_demo.silver.silver1_orders_clean

        UNION ALL

        SELECT
            'silver1_stores_clean',
            COUNT(*)
        FROM retail_demo.silver.silver1_stores_clean

        UNION ALL

        SELECT
            'quarantine_orders',
            COUNT(*)
        FROM retail_demo.silver.quarantine_orders

        ORDER BY table_name
    """)
)

✓ retail_demo.silver.silver1_customers_clean             577 rows | 12 columns
✓ retail_demo.silver.silver1_products_clean              166 rows | 12 columns
✓ retail_demo.silver.silver1_orders_clean                5,716 rows | 16 columns
✓ retail_demo.silver.silver1_stores_clean                75 rows | 8 columns
✓ retail_demo.silver.quarantine_orders                   284 rows | 20 columns


table_name,row_count
quarantine_orders,284
silver1_customers_clean,577
silver1_orders_clean,5716
silver1_products_clean,166
silver1_stores_clean,75


In [0]:
# CUSTOMER SCD2 VALIDATION
customer_scd2 = spark.table(
    "retail_demo.silver.dim_customer_scd2"
)

total_rows = customer_scd2.count()
distinct_customers = (
    customer_scd2
    .select("customer_id")
    .distinct()
    .count()
)
current_rows = (
    customer_scd2
    .filter(F.col("is_current") == True)
    .count()
)
historical_rows = (
    customer_scd2
    .filter(F.col("is_current") == False)
    .count()
)
null_surrogate_keys = (
    customer_scd2
    .filter(F.col("customer_sk").isNull())
    .count()
)
duplicate_surrogate_keys = (
    customer_scd2
    .groupBy("customer_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
incorrect_current_rows = (
    customer_scd2
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(
                F.col("is_current") == True,
                1
            ).otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") != 1)
    .count()
)
invalid_date_ranges = (
    customer_scd2
    .filter(
        F.col("effective_end_date").isNotNull()
        &
        (
            F.col("effective_end_date")
            < F.col("effective_start_date")
        )
    )
    .count()
)
current_not_open_ended = (
    customer_scd2
    .filter(
        (F.col("is_current") == True)
        &
        (
            F.col("effective_end_date")
            != F.to_date(F.lit("9999-12-31"))
        )
    )
    .count()
)

print("Total rows:", total_rows)
print("Distinct customer IDs:", distinct_customers)
print("Current rows:", current_rows)
print("Historical rows:", historical_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)
print("Customers with incorrect current-row count:",
      incorrect_current_rows)
print("Invalid date ranges:", invalid_date_ranges)
print("Current rows not open-ended:", current_not_open_ended)

Total rows: 2475
Distinct customer IDs: 2475
Current rows: 2475
Historical rows: 0
Null surrogate keys: 0
Duplicate surrogate keys: 0
Customers with incorrect current-row count: 0
Invalid date ranges: 0
Current rows not open-ended: 0


In [0]:
# PRODUCT SCD2 VALIDATION

product_scd2 = spark.table(
    "retail_demo.silver.dim_product_scd2"
)
total_rows = product_scd2.count()
distinct_products = (
    product_scd2
    .select("product_id")
    .distinct()
    .count()
)
current_rows = (
    product_scd2
    .filter(F.col("is_current") == True)
    .count()
)
historical_rows = (
    product_scd2
    .filter(F.col("is_current") == False)
    .count()
)
null_surrogate_keys = (
    product_scd2
    .filter(F.col("product_sk").isNull())
    .count()
)
duplicate_surrogate_keys = (
    product_scd2
    .groupBy("product_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
incorrect_current_rows = (
    product_scd2
    .groupBy("product_id")
    .agg(
        F.sum(
            F.when(
                F.col("is_current") == True,
                1
            ).otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") != 1)
    .count()
)
invalid_date_ranges = (
    product_scd2
    .filter(
        F.col("effective_end_date").isNotNull()
        &
        (
            F.col("effective_end_date")
            < F.col("effective_start_date")
        )
    )
    .count()
)
current_not_open_ended = (
    product_scd2
    .filter(
        (F.col("is_current") == True)
        &
        (
            F.col("effective_end_date")
            != F.to_date(F.lit("9999-12-31"))
        )
    )
    .count()
)

print("Total rows:", total_rows)
print("Distinct product IDs:", distinct_products)
print("Current rows:", current_rows)
print("Historical rows:", historical_rows)
print("Null surrogate keys:", null_surrogate_keys)
print("Duplicate surrogate keys:", duplicate_surrogate_keys)
print("Products with incorrect current-row count:",
      incorrect_current_rows)
print("Invalid date ranges:", invalid_date_ranges)
print("Current rows not open-ended:", current_not_open_ended)

Total rows: 980
Distinct product IDs: 800
Current rows: 800
Historical rows: 180
Null surrogate keys: 0
Duplicate surrogate keys: 0
Products with incorrect current-row count: 0
Invalid date ranges: 0
Current rows not open-ended: 0


In [0]:
# STORE DIMENSION VALIDATION

store_dim = spark.table(
    "retail_demo.silver.dim_store"
)
total_rows = store_dim.count()

distinct_stores = (
    store_dim
    .select("store_id")
    .distinct()
    .count()
)
null_store_ids = (
    store_dim
    .filter(F.col("store_id").isNull())
    .count()
)
duplicate_store_ids = (
    store_dim
    .groupBy("store_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print("Total rows:", total_rows)
print("Distinct store IDs:", distinct_stores)
print("Null store IDs:", null_store_ids)
print("Duplicate store IDs:", duplicate_store_ids)

print("\nSchema:")
store_dim.printSchema()

print("\nSample:")
display(store_dim.limit(10))

Total rows: 75
Distinct store IDs: 75
Null store IDs: 0
Duplicate store IDs: 0

Schema:
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)


Sample:


store_id,store_name,city,region,status
S019,Store_19,Mumbai,South,closed
S030,Store_30,Chennai,North,closed
S034,Store_34,Ahmedabad,South,active
S040,Store_40,Pune,East,active
S053,Store_53,Jaipur,South,active
S057,Store_57,Jaipur,Online,active
S012,Store_12,Gurugram,North,active
S039,Store_39,Mumbai,South,active
S066,Store_66,Gurugram,Online,closed
S024,Store_24,Hyderabad,North,closed


In [0]:
# GOLD FACT VALIDATION
fact = spark.table(
    "retail_demo.gold.fact_orders"
)
total_rows = fact.count()
distinct_orders = (
    fact
    .select("order_id")
    .distinct()
    .count()
)
duplicate_orders = (
    fact
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_order_ids = (
    fact
    .filter(F.col("order_id").isNull())
    .count()
)
null_customer_sk = (
    fact
    .filter(F.col("customer_sk").isNull())
    .count()
)
null_product_sk = (
    fact
    .filter(F.col("product_sk").isNull())
    .count()
)
null_store_ids = (
    fact
    .filter(F.col("store_id").isNull())
    .count()
)
null_quantity = (
    fact
    .filter(F.col("quantity").isNull())
    .count()
)

null_gross_amount = (
    fact
    .filter(F.col("gross_amount").isNull())
    .count()
)

metrics = fact.select(
    F.sum("quantity").alias("total_units"),
    F.sum("gross_amount").alias("total_revenue")
).collect()[0]

total_units = metrics["total_units"]
total_revenue = metrics["total_revenue"]

print("Total fact rows:", total_rows)
print("Distinct orders:", distinct_orders)
print("Duplicate order IDs:", duplicate_orders)

print("\nCritical key checks:")
print("Null order IDs:", null_order_ids)
print("Null customer_sk:", null_customer_sk)
print("Null product_sk:", null_product_sk)
print("Null store IDs:", null_store_ids)

print("\nMeasure checks:")
print("Null quantity:", null_quantity)
print("Null gross_amount:", null_gross_amount)

print("\nFact totals:")
print("Total units:", total_units)
print("Total revenue:", total_revenue)

fact_grain_valid = (
    total_rows == distinct_orders
    and duplicate_orders == 0
    and null_order_ids == 0
    and null_product_sk == 0
    and null_store_ids == 0
    and null_quantity == 0
    and null_gross_amount == 0
)

if fact_grain_valid:
    print(" GOLD FACT GRAIN: PASS")
else:
    print("GOLD FACT GRAIN: CHECK REQUIRED")

Total fact rows: 5716
Distinct orders: 5716
Duplicate order IDs: 0

Critical key checks:
Null order IDs: 0
Null customer_sk: 56
Null product_sk: 0
Null store IDs: 0

Measure checks:
Null quantity: 0
Null gross_amount: 0

Fact totals:
Total units: 19864
Total revenue: 803347391.80
 GOLD FACT GRAIN: PASS


In [0]:
# GOLD DAILY SALES VALIDATION
daily_sales = spark.table(
    "retail_demo.gold.gold_daily_sales"
)

days = daily_sales.count()

daily_metrics = daily_sales.select(
    F.sum("total_orders").alias("total_orders"),
    F.sum("total_units").alias("total_units"),
    F.sum("total_revenue").alias("total_revenue")
).collect()[0]

fact_metrics = spark.table(
    "retail_demo.gold.fact_orders"
).select(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("quantity").alias("total_units"),
    F.sum("gross_amount").alias("total_revenue")
).collect()[0]

print("Days:", days)

print("\nDaily Sales totals:")
print("Orders:", daily_metrics["total_orders"])
print("Units:", daily_metrics["total_units"])
print("Revenue:", daily_metrics["total_revenue"])

print("\nFact totals:")
print("Orders:", fact_metrics["total_orders"])
print("Units:", fact_metrics["total_units"])
print("Revenue:", fact_metrics["total_revenue"])

daily_reconciliation = (
    daily_metrics["total_orders"]
    == fact_metrics["total_orders"]
    and
    daily_metrics["total_units"]
    == fact_metrics["total_units"]
    and
    daily_metrics["total_revenue"]
    == fact_metrics["total_revenue"]
)

if daily_reconciliation:
    print(" DAILY SALES RECONCILIATION: PASS")
else:
    print(" DAILY SALES RECONCILIATION: CHECK REQUIRED")

print("\nSchema:")
daily_sales.printSchema()

Days: 24

Daily Sales totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80

Fact totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80
 DAILY SALES RECONCILIATION: PASS

Schema:
root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)
 |-- average_order_value: decimal(38,12) (nullable = true)



In [0]:
# GOLD CATEGORY SALES VALIDATION

category_sales = spark.table(
    "retail_demo.gold.gold_category_sales"
)

category_count = category_sales.count()

category_metrics = category_sales.select(
    F.sum("total_orders").alias("total_orders"),
    F.sum("total_units").alias("total_units"),
    F.sum("total_revenue").alias("total_revenue")
).collect()[0]

fact_metrics = spark.table(
    "retail_demo.gold.fact_orders"
).select(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("quantity").alias("total_units"),
    F.sum("gross_amount").alias("total_revenue")
).collect()[0]

print("Categories:", category_count)

print("\nCategory Sales totals:")
print("Orders:", category_metrics["total_orders"])
print("Units:", category_metrics["total_units"])
print("Revenue:", category_metrics["total_revenue"])

print("\nFact totals:")
print("Orders:", fact_metrics["total_orders"])
print("Units:", fact_metrics["total_units"])
print("Revenue:", fact_metrics["total_revenue"])

category_reconciliation = (
    category_metrics["total_orders"]
    == fact_metrics["total_orders"]
    and
    category_metrics["total_units"]
    == fact_metrics["total_units"]
    and
    category_metrics["total_revenue"]
    == fact_metrics["total_revenue"]
)
if category_reconciliation:
    print(" CATEGORY SALES RECONCILIATION: PASS")
else:
    print(" CATEGORY SALES RECONCILIATION: CHECK REQUIRED")

print("\nSchema:")
category_sales.printSchema()

print("\nCategory results:")
display(category_sales)

Categories: 7

Category Sales totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80

Fact totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80
 CATEGORY SALES RECONCILIATION: PASS

Schema:
root
 |-- product_category: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)


Category results:


product_category,total_orders,total_revenue,total_units
Home,1321,175889818.82,4624
Fashion,1218,170703562.45,4205
Grocery,1045,158045921.79,3620
Electronics,1040,141623368.40,3642
Beauty,984,141063023.45,3429
Unknown,96,13854391.62,301
null,12,2167305.27,43


In [0]:
#  GOLD SEGMENT SALES VALIDATION

print("=" * 80)
print("GOLD SEGMENT SALES VALIDATION")
print("=" * 80)

segment_sales = spark.table(
    "retail_demo.gold.gold_segment_sales"
)

segment_count = segment_sales.count()

segment_metrics = segment_sales.select(
    F.sum("total_orders").alias("total_orders"),
    F.sum("total_revenue").alias("total_revenue")
).collect()[0]

fact_metrics = spark.table(
    "retail_demo.gold.fact_orders"
).select(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("gross_amount").alias("total_revenue")
).collect()[0]

print("Segments:", segment_count)

print("\nSegment Sales totals:")
print("Orders:", segment_metrics["total_orders"])
print("Revenue:", segment_metrics["total_revenue"])

print("\nFact totals:")
print("Orders:", fact_metrics["total_orders"])
print("Revenue:", fact_metrics["total_revenue"])

segment_reconciliation = (
    segment_metrics["total_orders"]
    == fact_metrics["total_orders"]
    and
    segment_metrics["total_revenue"]
    == fact_metrics["total_revenue"]
)

required_columns = {
    "customer_segment",
    "unique_customers",
    "total_orders",
    "total_revenue"
}

actual_columns = set(segment_sales.columns)

columns_present = required_columns.issubset(actual_columns)

print("\nRequired columns present:", columns_present)

if segment_reconciliation and columns_present:
    print(" SEGMENT SALES RECONCILIATION: PASS")
else:
    print("SEGMENT SALES RECONCILIATION: CHECK REQUIRED")
print("\nSchema:")
segment_sales.printSchema()

print("\nSegment results:")
display(segment_sales)

GOLD SEGMENT SALES VALIDATION
Segments: 5

Segment Sales totals:
Orders: 5716
Revenue: 803347391.80

Fact totals:
Orders: 5716
Revenue: 803347391.80

Required columns present: True
 SEGMENT SALES RECONCILIATION: PASS

Schema:
root
 |-- customer_segment: string (nullable = true)
 |-- unique_customers: long (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: decimal(28,2) (nullable = true)


Segment results:


customer_segment,unique_customers,total_orders,total_revenue
Regular,591,1491,206240277.72
Platinum,591,1427,200914743.92
Gold,573,1391,197266776.71
Silver,566,1351,192728337.84
null,23,56,6197255.61


In [0]:
#  GOLD REGION SALES VALIDATION
region_sales = spark.table(
    "retail_demo.gold.gold_region_sales"
)

region_count = region_sales.count()

region_metrics = region_sales.select(
    F.sum("total_orders").alias("total_orders"),
    F.sum("total_units").alias("total_units"),
    F.sum("total_revenue").alias("total_revenue")
).collect()[0]

fact_metrics = spark.table(
    "retail_demo.gold.fact_orders"
).select(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("quantity").alias("total_units"),
    F.sum("gross_amount").alias("total_revenue")
).collect()[0]

print("Regions:", region_count)

print("\nRegion Sales totals:")
print("Orders:", region_metrics["total_orders"])
print("Units:", region_metrics["total_units"])
print("Revenue:", region_metrics["total_revenue"])

print("\nFact totals:")
print("Orders:", fact_metrics["total_orders"])
print("Units:", fact_metrics["total_units"])
print("Revenue:", fact_metrics["total_revenue"])

region_reconciliation = (
    region_metrics["total_orders"]
    == fact_metrics["total_orders"]
    and
    region_metrics["total_units"]
    == fact_metrics["total_units"]
    and
    region_metrics["total_revenue"]
    == fact_metrics["total_revenue"]
)

required_columns = {
    "store_region",
    "total_orders",
    "total_revenue",
    "total_units"
}

columns_present = required_columns.issubset(
    set(region_sales.columns)
)

print("\nRequired columns present:", columns_present)
if region_reconciliation and columns_present:
    print(" REGION SALES RECONCILIATION: PASS")
else:
    print(" REGION SALES RECONCILIATION: CHECK REQUIRED")
print("=" * 80)
print("\nSchema:")
region_sales.printSchema()
print("\nRegion results:")
display(region_sales)

Regions: 5

Region Sales totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80

Fact totals:
Orders: 5716
Units: 19864
Revenue: 803347391.80

Required columns present: True
 REGION SALES RECONCILIATION: PASS

Schema:
root
 |-- store_region: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)


Region results:


store_region,total_orders,total_revenue,total_units
Online,1398,198733595.58,4823
South,1325,189059108.26,4626
North,1122,158536875.73,3928
West,1067,148379720.94,3767
East,804,108638091.29,2720


In [0]:
# DELTA TABLE HISTORY VALIDATION

fact_history = spark.sql("""
    DESCRIBE HISTORY retail_demo.gold.fact_orders
""")

history_count = fact_history.count()

print("History versions:", history_count)

print("\nHistory:")
display(
    fact_history.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

history_valid = history_count >= 1


if history_valid:
    print(" DELTA TABLE HISTORY: PASS")
else:
    print(" DELTA TABLE HISTORY: CHECK REQUIRED")


History versions: 3

History:


version,timestamp,operation,operationMetrics
2,2026-08-13T02:33:22.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 298749, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"
1,2026-08-12T20:16:33.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 298749, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"
0,2026-08-12T14:45:48.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"


 DELTA TABLE HISTORY: PASS


In [0]:
# ============================================================
# CELL 13 — DELTA TIME TRAVEL VALIDATION
# ============================================================

print("=" * 80)
print("DELTA TIME TRAVEL VALIDATION")
print("=" * 80)

current_metrics = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        SUM(quantity) AS total_units,
        SUM(gross_amount) AS total_revenue
    FROM retail_demo.gold.fact_orders
""").collect()[0]

version_0_metrics = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        SUM(quantity) AS total_units,
        SUM(gross_amount) AS total_revenue
    FROM retail_demo.gold.fact_orders
    VERSION AS OF 0
""").collect()[0]

print("Current fact:")
print("  Rows:", current_metrics["total_rows"])
print("  Distinct orders:", current_metrics["distinct_orders"])
print("  Units:", current_metrics["total_units"])
print("  Revenue:", current_metrics["total_revenue"])

print("\nVersion 0:")
print("  Rows:", version_0_metrics["total_rows"])
print("  Distinct orders:", version_0_metrics["distinct_orders"])
print("  Units:", version_0_metrics["total_units"])
print("  Revenue:", version_0_metrics["total_revenue"])

time_travel_valid = (
    version_0_metrics["total_rows"] == current_metrics["total_rows"]
    and
    version_0_metrics["distinct_orders"] == current_metrics["distinct_orders"]
    and
    version_0_metrics["total_units"] == current_metrics["total_units"]
    and
    version_0_metrics["total_revenue"] == current_metrics["total_revenue"]
)
if time_travel_valid:
    print(" DELTA TIME TRAVEL: PASS")
else:
    print(" DELTA TIME TRAVEL: CHECK REQUIRED")

DELTA TIME TRAVEL VALIDATION
Current fact:
  Rows: 5716
  Distinct orders: 5716
  Units: 19864
  Revenue: 803347391.80

Version 0:
  Rows: 5716
  Distinct orders: 5716
  Units: 19864
  Revenue: 803347391.80
 DELTA TIME TRAVEL: PASS


In [0]:
#  DELTA SCHEMA EVOLUTION VALIDATION
schema_demo = spark.table(
    "retail_demo.gold.fact_orders_schema_evolution_demo"
)

column_names = [
    field.name
    for field in schema_demo.schema.fields
]

coupon_code_present = "coupon_code" in column_names

final_rows = schema_demo.count()

schema_history = spark.sql("""
    DESCRIBE HISTORY
    retail_demo.gold.fact_orders_schema_evolution_demo
""")
history_versions = schema_history.count()
print("coupon_code present:", coupon_code_present)
print("Final rows:", final_rows)
print("History versions:", history_versions)
print("\nFinal schema:")
schema_demo.printSchema()
print("\nDelta history:")
display(
    schema_history.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

schema_evolution_valid = (
    coupon_code_present
    and final_rows == 120
    and history_versions >= 2
)
if schema_evolution_valid:
    print(" DELTA SCHEMA EVOLUTION: PASS")
else:
    print(" DELTA SCHEMA EVOLUTION: CHECK REQUIRED")

coupon_code present: True
Final rows: 120
History versions: 6

Final schema:
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_sk: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- coupon_code: string (nullable = true)


Delta history:


version,timestamp,operation,operationMetrics
5,2026-08-13T02:34:04.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6704)"
4,2026-08-13T02:34:01.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 20492, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"
3,2026-08-12T20:17:24.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6700)"
2,2026-08-12T20:17:20.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 20496, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"
1,2026-08-12T16:07:47.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6704)"
0,2026-08-12T16:07:44.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"


 DELTA SCHEMA EVOLUTION: PASS


In [0]:

# FINAL CROSS-LAYER VALIDATION

from decimal import Decimal
EXPECTED_REVENUE = Decimal("803347391.80")
EXPECTED_UNITS = 19864
EXPECTED_ORDERS = 5716

# 1. BRONZE

bronze_expected = {
    "customers_bronze": 2560,
    "orders_bronze": 12180,
    "products_bronze": 830,
    "stores_bronze": 80,
    "bronze_customers_incremental": 630,
    "bronze_orders_incremental": 6135,
    "bronze_products_incremental": 180
}
bronze_pass = True
print("\nBRONZE")
for table_name, expected_count in bronze_expected.items():
    actual_count = spark.table(
        f"retail_demo.raw.{table_name}"
    ).count()
    passed = actual_count == expected_count
    bronze_pass = bronze_pass and passed
    print(
        f"{'✓' if passed else '✗'} "
        f"{table_name:<35}"
        f"expected={expected_count:,} "
        f"actual={actual_count:,}"
    )

# 2. SILVER STAGE 1
silver_expected = {
    "silver1_customers_clean": 577,
    "silver1_products_clean": 166,
    "silver1_orders_clean": 5716,
    "silver1_stores_clean": 75,
    "quarantine_orders": 284
}
silver_pass = True
print("\nSILVER STAGE 1")
for table_name, expected_count in silver_expected.items():

    actual_count = spark.table(
        f"retail_demo.silver.{table_name}"
    ).count()
    passed = actual_count == expected_count
    silver_pass = silver_pass and passed
    print(
        f"{'✓' if passed else '✗'} "
        f"{table_name:<35}"
        f"expected={expected_count:,} "
        f"actual={actual_count:,}"
    )

# 3. CUSTOMER SCD2

customer_scd2 = spark.table(
    "retail_demo.silver.dim_customer_scd2"
)
customer_total = customer_scd2.count()
customer_distinct = (
    customer_scd2
    .select("customer_id")
    .distinct()
    .count()
)
customer_current = (
    customer_scd2
    .filter(F.col("is_current") == True)
    .count()
)
customer_historical = (
    customer_scd2
    .filter(F.col("is_current") == False)
    .count()
)
customer_null_sk = (
    customer_scd2
    .filter(F.col("customer_sk").isNull())
    .count()
)
customer_scd2_pass = (
    customer_total == 3067
    and customer_distinct == 2719
    and customer_current == 2719
    and customer_historical == 348
    and customer_null_sk == 0
)
print(
    f"\n{'✓' if customer_scd2_pass else '✗'} "
    f"Customer SCD2"
)
# 4. PRODUCT SCD2

product_scd2 = spark.table(
    "retail_demo.silver.dim_product_scd2"
)
product_total = product_scd2.count()
product_distinct = (
    product_scd2
    .select("product_id")
    .distinct()
    .count()
)

product_current = (
    product_scd2
    .filter(F.col("is_current") == True)
    .count()
)

product_historical = (
    product_scd2
    .filter(F.col("is_current") == False)
    .count()
)

product_null_sk = (
    product_scd2
    .filter(F.col("product_sk").isNull())
    .count()
)

product_scd2_pass = (
    product_total == 980
    and product_distinct == 800
    and product_current == 800
    and product_historical == 180
    and product_null_sk == 0
)

print(
    f"{'' if product_scd2_pass else '✗'} "
    f"Product SCD2"
)

# 5. STORE DIMENSION

store_dim = spark.table(
    "retail_demo.silver.dim_store"
)

store_pass = (
    store_dim.count() == 75
    and
    store_dim.select("store_id").distinct().count() == 75
)

print(
    f"{'' if store_pass else '✗'} "
    f"Store Dimension"
)

# 6. GOLD FACT

fact = spark.table(
    "retail_demo.gold.fact_orders"
)
fact_metrics = fact.select(
    F.count("*").alias("rows"),
    F.countDistinct("order_id").alias("orders"),
    F.sum("quantity").alias("units"),
    F.sum("gross_amount").alias("revenue")
).collect()[0]
duplicate_orders = (
    fact
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

gold_fact_pass = (
    fact_metrics["rows"] == EXPECTED_ORDERS
    and
    fact_metrics["orders"] == EXPECTED_ORDERS
    and
    duplicate_orders == 0
    and
    fact_metrics["units"] == EXPECTED_UNITS
    and
    fact_metrics["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'✓' if gold_fact_pass else '✗'} "
    f"Gold Fact"
)
# 7. GOLD DAILY SALES
daily = spark.table(
    "retail_demo.gold.gold_daily_sales"
)

daily_metrics = daily.select(
    F.count("*").alias("days"),
    F.sum("total_orders").alias("orders"),
    F.sum("total_units").alias("units"),
    F.sum("total_revenue").alias("revenue")
).collect()[0]

daily_pass = (
    daily_metrics["days"] == 24
    and
    daily_metrics["orders"] == EXPECTED_ORDERS
    and
    daily_metrics["units"] == EXPECTED_UNITS
    and
    daily_metrics["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'' if daily_pass else '✗'} "
    f"Gold Daily Sales"
)
# 8. GOLD CATEGORY SALES

category = spark.table(
    "retail_demo.gold.gold_category_sales"
)

category_metrics = category.select(
    F.count("*").alias("categories"),
    F.sum("total_orders").alias("orders"),
    F.sum("total_units").alias("units"),
    F.sum("total_revenue").alias("revenue")
).collect()[0]

category_pass = (
    category_metrics["categories"] == 7
    and
    category_metrics["orders"] == EXPECTED_ORDERS
    and
    category_metrics["units"] == EXPECTED_UNITS
    and
    category_metrics["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'' if category_pass else '✗'} "
    f"Gold Category Sales"
)


# 9. GOLD SEGMENT SALES
segment = spark.table(
    "retail_demo.gold.gold_segment_sales"
)

segment_metrics = segment.select(
    F.count("*").alias("segments"),
    F.sum("total_orders").alias("orders"),
    F.sum("total_revenue").alias("revenue")
).collect()[0]

segment_pass = (
    segment_metrics["segments"] == 5
    and
    segment_metrics["orders"] == EXPECTED_ORDERS
    and
    segment_metrics["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'' if segment_pass else '✗'} "
    f"Gold Segment Sales"
)
# 10. GOLD REGION SALES

region = spark.table(
    "retail_demo.gold.gold_region_sales"
)

region_metrics = region.select(
    F.count("*").alias("regions"),
    F.sum("total_orders").alias("orders"),
    F.sum("total_units").alias("units"),
    F.sum("total_revenue").alias("revenue")
).collect()[0]

region_pass = (
    region_metrics["regions"] == 5
    and
    region_metrics["orders"] == EXPECTED_ORDERS
    and
    region_metrics["units"] == EXPECTED_UNITS
    and
    region_metrics["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'' if region_pass else '✗'} "
    f"Gold Region Sales"
)
# 11. DELTA HISTORY
history = spark.sql("""
    DESCRIBE HISTORY retail_demo.gold.fact_orders
""")

history_pass = history.count() >= 1

print(
    f"{'✓' if history_pass else '✗'} "
    f"Delta Table History"
)

# 12. DELTA TIME TRAVEL

time_travel = spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT order_id) AS orders,
        SUM(quantity) AS units,
        SUM(gross_amount) AS revenue
    FROM retail_demo.gold.fact_orders
    VERSION AS OF 0
""").collect()[0]

time_travel_pass = (
    time_travel["rows"] == EXPECTED_ORDERS
    and
    time_travel["orders"] == EXPECTED_ORDERS
    and
    time_travel["units"] == EXPECTED_UNITS
    and
    time_travel["revenue"] == EXPECTED_REVENUE
)

print(
    f"{'✓' if time_travel_pass else '✗'} "
    f"Delta Time Travel"
)
# 13. DELTA SCHEMA EVOLUTION

schema_demo = spark.table(
    "retail_demo.gold.fact_orders_schema_evolution_demo"
)

schema_columns = {
    field.name
    for field in schema_demo.schema.fields
}

schema_history = spark.sql("""
    DESCRIBE HISTORY
    retail_demo.gold.fact_orders_schema_evolution_demo
""")

schema_evolution_pass = (
    "coupon_code" in schema_columns
    and
    schema_demo.count() == 120
    and
    schema_history.count() >= 2
)

print(
    f"{'' if schema_evolution_pass else '✗'} "
    f"Delta Schema Evolution"
)
# 14. FINAL RESULT

overall_pass = (
    bronze_pass
    and silver_pass
    and customer_scd2_pass
    and product_scd2_pass
    and store_pass
    and gold_fact_pass
    and daily_pass
    and category_pass
    and segment_pass
    and region_pass
    and history_pass
    and time_travel_pass
    and schema_evolution_pass
)
if overall_pass:
    print("OVERALL PROJECT STATUS: PASS")
    print("All project validations passed successfully.")
else:
    print("⚠ OVERALL PROJECT STATUS: CHECK REQUIRED")

print("\nFINAL PROJECT METRICS")
print("-" * 80)
print("Fact orders      :", fact_metrics["rows"])
print("Distinct orders  :", fact_metrics["orders"])
print("Total units      :", fact_metrics["units"])
print("Total revenue    :", fact_metrics["revenue"])
print("Customer SCD2    :", customer_total)
print("Product SCD2     :", product_total)


BRONZE
✓ customers_bronze                   expected=2,560 actual=2,560
✓ orders_bronze                      expected=12,180 actual=12,180
✓ products_bronze                    expected=830 actual=830
✓ stores_bronze                      expected=80 actual=80
✓ bronze_customers_incremental       expected=630 actual=630
✓ bronze_orders_incremental          expected=6,135 actual=6,135
✓ bronze_products_incremental        expected=180 actual=180

SILVER STAGE 1
✓ silver1_customers_clean            expected=577 actual=577
✓ silver1_products_clean             expected=166 actual=166
✓ silver1_orders_clean               expected=5,716 actual=5,716
✓ silver1_stores_clean               expected=75 actual=75
✓ quarantine_orders                  expected=284 actual=284

✗ Customer SCD2
 Product SCD2
 Store Dimension
✓ Gold Fact
 Gold Daily Sales
 Gold Category Sales
 Gold Segment Sales
 Gold Region Sales
✓ Delta Table History
✓ Delta Time Travel
 Delta Schema Evolution
⚠ OVERALL PROJECT STATUS: 